# Practical 7: Real-Time Object Detection for Foliar Pests & Lesions
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit IV (Object Detection & Localization) | **CO:** CO4  
**Problem Statement Context:** SIH26131 — Stage 2: In-Field Pest Localization & Bounding Box Prediction  

---

## 1. Single-Stage Detection Principles
Single-stage detectors (YOLO, SSD, RetinaNet) formulate object detection as a dense spatial regression problem:
1. Divide input image into an $S 	imes S$ grid.
2. For each grid cell, predict $B$ bounding boxes with coordinates $(t_x, t_y, t_w, t_h)$, an objectness confidence score $P(Object)$, and class conditional probabilities $P(Class_c | Object)$.
3. Post-process candidate boxes using **Non-Maximum Suppression (NMS)** to eliminate redundant overlapping predictions based on Intersection over Union (IoU):

$$	ext{IoU}(A, B) = rac{	ext{Area}(A \cap B)}{	ext{Area}(A \cup B)}$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.detector import compute_iou, non_max_suppression_scratch, box_xyxy_to_xywh

## 2. Load Foliar Pest Detection Samples

In [ ]:
det_dir = REPO_ROOT / "data" / "samples" / "detection"
img_path = det_dir / "images" / "det_000.jpg"
image = Image.open(img_path).convert("RGB")
img_w, img_h = image.size

print(f"[✓] Loaded detection test image: {img_path.name} ({img_w}x{img_h})")

## 3. Simulate Candidate Bounding Boxes & Apply NMS

In [ ]:
# Candidate raw bounding boxes [x1, y1, x2, y2]
raw_boxes = torch.tensor([
    [100.0, 110.0, 190.0, 205.0],  # Box 1: Aphid cluster
    [105.0, 115.0, 185.0, 200.0],  # Box 2: Duplicate of Box 1 (IoU ~ 0.85)
    [98.0,  108.0, 192.0, 210.0],  # Box 3: Duplicate of Box 1 (IoU ~ 0.90)
    [240.0, 160.0, 320.0, 250.0],  # Box 4: Blight spot
    [245.0, 162.0, 315.0, 248.0],  # Box 5: Duplicate of Box 4
], dtype=torch.float32)

raw_scores = torch.tensor([0.92, 0.74, 0.65, 0.88, 0.58], dtype=torch.float32)
labels = ["aphid", "aphid", "aphid", "blight_spot", "blight_spot"]

print(f"Total raw proposals before NMS: {len(raw_boxes)}")
keep_indices = non_max_suppression_scratch(raw_boxes, raw_scores, iou_threshold=0.45, score_threshold=0.5)
print(f"Retained box indices after NMS: {keep_indices.tolist()}")

## 4. Visualizing Raw Proposals vs. Suppressed Detections

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Left: Raw redundant boxes
axes[0].imshow(image)
axes[0].set_title(f"Before NMS: {len(raw_boxes)} Overlapping Proposals")
axes[0].axis("off")
for box, score, lbl in zip(raw_boxes, raw_scores, labels):
    x1, y1, x2, y2 = box.tolist()
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1 - 5, f"{lbl} {score:.2f}", color='yellow', fontsize=10, backgroundcolor='black')

# Right: Clean NMS output
axes[1].imshow(image)
axes[1].set_title(f"After NMS: {len(keep_indices)} Pruned Detections")
axes[1].axis("off")
for idx in keep_indices:
    box = raw_boxes[idx].tolist()
    score = raw_scores[idx].item()
    lbl = labels[idx]
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=3, edgecolor='lime', facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x1, y1 - 8, f"{lbl} {score:.2f}", color='white', fontsize=11, weight='bold', backgroundcolor='green')

plt.tight_layout()
plt.show()

## 5. Practical Summary
- Non-Maximum Suppression is essential to collapse multiple overlapping multi-scale anchor detections into a single precise bounding box per individual foliar pest or lesion.